# Colab: MobileNetV4ConvSmall FSD50K Pretraining

This notebook pretrains a MobileNetV4 Conv Small audio classifier on FSD50K precomputed mel features (`npy_fsd50k.zip`) stored on Google Drive.

**Setup Instructions:**
1. Upload `npy_fsd50k.zip` to Google Drive
2. Keep `FSD50K.ground_truth` on Drive (for labels/splits)
3. Update the Drive path variables below if your layout is different
4. Ensure GPU is enabled: Runtime -> Change runtime type -> GPU
5. Run all cells in order

## Step 1: Mount Google Drive and Configure Paths

In [ ]:
from google.colab import drive
from pathlib import Path
import sys

# Mount Google Drive
drive.mount('/content/drive')
print('Google Drive mounted successfully!')

Mounted at /content/drive
Google Drive mounted successfully!


In [ ]:
import zipfile

DRIVE_ROOT = Path('/content/drive/MyDrive')

# Ground-truth CSV + vocabulary location (labels/splits)
DRIVE_FSD50K_GROUND_TRUTH_PATH = DRIVE_ROOT / 'SoundPulse' / 'data' / 'raw' / 'FSDK50_dataset' / 'FSD50K.ground_truth'

# Uploaded features archive on Drive
DRIVE_NPY_FSD50K_ZIP_PATH = DRIVE_ROOT / 'SoundPulse' / 'npy_fsd50k.zip'

# Path to eval zip folder
DRIVE_EVAL_FSD50K_ZIP_PATH = DRIVE_ROOT / 'SoundPulse' / 'npy_fsd50k_eval'/ 'npy_fsd50k_eval.zip'

# Extraction target in local Colab storage for fast training I/O
LOCAL_NPY_FSD50K_PATH_BASE = Path('/npy_fsd50k') # Renamed to base path
LOCAL_NPY_FSD50K_PATH_BASE.mkdir(parents=True, exist_ok=True)

# Adjust LOCAL_NPY_FSD50K_PATH to point to the actual content directory inside the zip
LOCAL_NPY_FSD50K_PATH = LOCAL_NPY_FSD50K_PATH_BASE / 'npy_fsd50k'
LOCAL_NPY_FSD50K_PATH.mkdir(parents=True, exist_ok=True) # Ensure the actual target path exists

# Evaluation data path for fast evaluation I/O
LOCAL_EVAL_NPY_FSD50K_PATH_BASE = Path('/npy_fsd50k_eval') # Renamed to base path
LOCAL_EVAL_NPY_FSD50K_PATH_BASE.mkdir(parents=True, exist_ok=True)

# Adjust LOCAL_EVAL_NPY_FSD50K_PATH to point to the actual content directory inside the zip
LOCAL_EVAL_NPY_FSD50K_PATH = LOCAL_EVAL_NPY_FSD50K_PATH_BASE / 'npy_fsd50k_eval'
LOCAL_EVAL_NPY_FSD50K_PATH.mkdir(parents=True, exist_ok=True) # Ensure the actual target path exists

# Output directories
MODEL_OUTPUT_DRIVE_PATH = DRIVE_ROOT / 'SoundPulse_outputs' / 'mobilenetv4_conv_small_fsd50k_pretrain'
MODEL_OUTPUT_DRIVE_PATH.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_LOCAL_PATH = Path('/content/model_outputs')
MODEL_OUTPUT_LOCAL_PATH.mkdir(parents=True, exist_ok=True)

print(f'Ground-truth path: {DRIVE_FSD50K_GROUND_TRUTH_PATH}')
print(f'Feature zip path: {DRIVE_NPY_FSD50K_ZIP_PATH}')
print(f'Eval audio features path: {DRIVE_EVAL_FSD50K_ZIP_PATH}') # New print
print(f'Extracted feature base dir: {LOCAL_NPY_FSD50K_PATH_BASE}') # Updated print
print(f'Extracted feature content dir: {LOCAL_NPY_FSD50K_PATH}') # New print
print(f'Model output (local): {MODEL_OUTPUT_LOCAL_PATH}')
print(f'Model output (Drive): {MODEL_OUTPUT_DRIVE_PATH}')

assert DRIVE_FSD50K_GROUND_TRUTH_PATH.exists(), f'FSD50K.ground_truth not found at {DRIVE_FSD50K_GROUND_TRUTH_PATH}'
assert DRIVE_NPY_FSD50K_ZIP_PATH.exists(), f'npy_fsd50k.zip not found at {DRIVE_NPY_FSD50K_ZIP_PATH}'
assert DRIVE_EVAL_FSD50K_ZIP_PATH.exists(), f'FSD50K eval features not found at {DRIVE_EVAL_FSD50K_ZIP_PATH}' # New assert

# Extract zip once (skip if already extracted)
# Marker file now in the base path, as extractall happens there
marker_file = LOCAL_NPY_FSD50K_PATH_BASE / '.extract_done'

# Force re-extraction if marker exists but no .npy files are found in the *actual* content dir
if marker_file.exists() and not list(LOCAL_NPY_FSD50K_PATH.glob('*.npy')):
    print(f'Found marker file "{marker_file}" but no .npy files in {LOCAL_NPY_FSD50K_PATH}. Removing marker to force re-extraction.')
    marker_file.unlink()

if not marker_file.exists():
    print(f'Extracting npy_fsd50k.zip to local runtime storage at {LOCAL_NPY_FSD50K_PATH_BASE}...') # Updated print
    with zipfile.ZipFile(DRIVE_NPY_FSD50K_ZIP_PATH, 'r') as archive:
        archive.extractall(LOCAL_NPY_FSD50K_PATH_BASE) # Extract to the base path
    marker_file.write_text('ok', encoding='utf-8')
else:
    print('Using previously extracted npy features (marker found).')

npy_files = list(LOCAL_NPY_FSD50K_PATH.glob('*.npy'))
print(f'✓ Found {len(npy_files)} .npy feature files')

# Extract eval zip once (skip if already extracted)
marker_file = LOCAL_EVAL_NPY_FSD50K_PATH_BASE / '.extract_done'

# Force re-extraction if marker exists but no .npy files are found in the *actual* content dir
if marker_file.exists() and not list(LOCAL_EVAL_NPY_FSD50K_PATH.glob('*.npy')):
    print(f'Found marker file "{marker_file}" but no .npy files in {LOCAL_EVAL_NPY_FSD50K_PATH}. Removing marker to force re-extraction.')
    marker_file.unlink()

if not marker_file.exists():
    print(f'Extracting npy_fsd50k_eval.zip to local runtime storage at {LOCAL_NPY_FSD50K_PATH_BASE}...') # Updated print
    with zipfile.ZipFile(DRIVE_EVAL_FSD50K_ZIP_PATH, 'r') as archive:
        archive.extractall(LOCAL_EVAL_NPY_FSD50K_PATH_BASE) # Extract to the base path
    marker_file.write_text('ok', encoding='utf-8')
else:
    print('Using previously extracted npy features (marker found).')

npy_files = list(LOCAL_EVAL_NPY_FSD50K_PATH.glob('*.npy'))
print(f'✓ Found {len(npy_files)} .npy eval feature files')

Ground-truth path: /content/drive/MyDrive/SoundPulse/data/raw/FSDK50_dataset/FSD50K.ground_truth
Feature zip path: /content/drive/MyDrive/SoundPulse/npy_fsd50k.zip
Eval audio features path: /content/drive/MyDrive/SoundPulse/npy_fsd50k_eval/npy_fsd50k_eval.zip
Extracted feature base dir: /npy_fsd50k
Extracted feature content dir: /npy_fsd50k/npy_fsd50k
Model output (local): /content/model_outputs
Model output (Drive): /content/drive/MyDrive/SoundPulse_outputs/mobilenetv4_conv_small_fsd50k_pretrain
Extracting npy_fsd50k.zip to local runtime storage at /npy_fsd50k...
✓ Found 40966 .npy feature files
Extracting npy_fsd50k_eval.zip to local runtime storage at /npy_fsd50k...
✓ Found 10231 .npy eval feature files


## Step 2: Install Dependencies

In [ ]:
%pip install -q tf-models-official tf-keras scikit-learn pandas numpy
print('Dependencies installed!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 112.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.4 MB/s eta 0:00:00
Dependencies installed!


In [ ]:
# Check GPU availability
import tensorflow as tf

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')
if not tf.config.list_physical_devices('GPU'):
    print('⚠️ WARNING: No GPU detected. Training will be slow. Enable GPU in Runtime → Change runtime type')

TensorFlow version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Step 3: Setup and Configuration

In [ ]:
from __future__ import annotations

import json
import numpy as np
import pandas as pd
import tensorflow as tf
import tf_keras as keras
from official.vision.modeling.backbones import mobilenet
from sklearn.metrics import average_precision_score, f1_score, hamming_loss, label_ranking_average_precision_score

# For Colab: use local output during training, then copy to Drive
MODEL_OUTPUT_DIR = MODEL_OUTPUT_LOCAL_PATH

# Runtime acceleration toggles
USE_MIXED_PRECISION = True
ENABLE_XLA = True

if ENABLE_XLA:
    tf.config.optimizer.set_jit(True)
if USE_MIXED_PRECISION:
    keras.mixed_precision.set_global_policy('mixed_float16')

# Training configuration
MODEL_ID = 'MobileNetV4ConvSmall'
SAMPLE_RATE = 16000
WINDOW_DURATION_MS = 960
WINDOW_SAMPLES = int(SAMPLE_RATE * WINDOW_DURATION_MS / 1000)

# Only the mel feature contract matters here; the notebook does not re-run preprocessing.
PREPROCESSING_CONFIG = {
    'mel': {'n_mels': 96},
}
EXPECTED_MEL_BINS = PREPROCESSING_CONFIG['mel']['n_mels']
TIME_FRAMES = 128
IMAGE_SIZE = (EXPECTED_MEL_BINS, TIME_FRAMES)
EMBEDDING_DIM = 512
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-5
PREDICTION_THRESHOLD = 0.5
AUTOTUNE = tf.data.AUTOTUNE

# Preprocessing summary for quick sanity checks in the notebook.
print('Configuration loaded')
print({
    'model_id': MODEL_ID,
    'sample_rate': SAMPLE_RATE,
    'preprocessing_config': PREPROCESSING_CONFIG,
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'mixed_precision_policy': keras.mixed_precision.global_policy().name,
    'xla_enabled': ENABLE_XLA,
    'model_output_dir': str(MODEL_OUTPUT_DIR),
})


Configuration loaded
{'model_id': 'MobileNetV4ConvSmall', 'sample_rate': 16000, 'image_size': (256, 256), 'batch_size': 32, 'epochs': 20, 'mixed_precision_policy': 'mixed_float16', 'xla_enabled': True, 'model_output_dir': '/content/model_outputs'}


In [ ]:
import numpy as np
from pathlib import Path

sample_feature_dir = LOCAL_NPY_FSD50K_PATH
expected_mels = PREPROCESSING_CONFIG['mel']['n_mels']

for feature_file in list(sample_feature_dir.glob('*.npy'))[:5]:
    feature = np.load(feature_file)
    print(feature_file.name, feature.shape, feature.dtype, feature.min(), feature.max(), np.mean(feature))
    if feature.ndim == 2 and expected_mels not in feature.shape:
        raise ValueError(
            f'{feature_file.name} does not match the expected mel-bin contract of {expected_mels} bins.'
        )


10000.npy (96, 14) float16 -73.1 0.0 -18.92
100005.npy (96, 13) float16 -80.0 0.0 -52.88
100006.npy (96, 26) float16 -80.0 0.0 -39.1
100007.npy (96, 26) float16 -80.0 0.0 -45.56
10001.npy (96, 16) float16 -80.0 0.0 -40.2


## Step 4: Load FSD50K Dataset

In [ ]:
def load_fsd50k_vocabulary(vocabulary_path: Path) -> pd.DataFrame:
    vocabulary = pd.read_csv(vocabulary_path, header=None, names=['index', 'label', 'mid'])
    vocabulary = vocabulary.sort_values('index').reset_index(drop=True)
    assert vocabulary['index'].tolist() == list(range(len(vocabulary))), 'FSD50K vocabulary indices must be contiguous.'
    return vocabulary


def labels_to_multihot(labels: str) -> np.ndarray:
    target = np.zeros(len(LABEL_ORDER), dtype=np.float32)
    for label in labels.split(','):
        index = LABEL_TO_INDEX.get(label)
        if index is not None:
            target[index] = 1.0
    return target


def load_fsd50k_split(csv_path: Path, data_root: Path, split_overrides: dict[str | None, str]) -> pd.DataFrame:
    frame = pd.read_csv(csv_path)
    if 'split' not in frame.columns:
        frame['split'] = None
    frame['split_name'] = frame['split'].map(split_overrides)
    frame = frame[frame['split_name'].notna()].copy()

    # Ensure fname is a string before path construction
    frame['fname_str'] = frame['fname'].astype(str)

    frame['data_path'] = frame['fname_str'].map(lambda fname_str: str(data_root / f'{fname_str}.npy'))

    # Debugging prints
    print(f'\n--- Debugging load_fsd50k_split for {csv_path.name} ---')
    print('First 5 generated data_paths (before existence check):')
    for i, path in enumerate(frame['data_path'].head()):
        print(f'  {path}')

    # Filter out non-existent files
    initial_count = len(frame)
    frame = frame[frame['data_path'].map(lambda value: Path(value).exists())].reset_index(drop=True)
    if len(frame) < initial_count:
        print(f'WARNING: {initial_count - len(frame)} files from {csv_path.name} did not exist at their specified data_path.')
    print('--- End Debugging ---\n')

    frame['target'] = frame['labels'].map(labels_to_multihot)
    return frame[['fname', 'labels', 'mids', 'split_name', 'data_path', 'target']]

# Load vocabulary + label mapping
FSD50K_GROUND_TRUTH_DIR = DRIVE_FSD50K_GROUND_TRUTH_PATH
FSD50K_FEATURE_DIR = LOCAL_NPY_FSD50K_PATH

vocabulary_df = load_fsd50k_vocabulary(FSD50K_GROUND_TRUTH_DIR / 'vocabulary.csv')
LABEL_ORDER = vocabulary_df['label'].tolist()
LABEL_TO_INDEX = {label: index for index, label in enumerate(LABEL_ORDER)}

print(f'Loaded {len(LABEL_ORDER)} audio classes')

Loaded 200 audio classes


In [ ]:
# Load train/validation/test splits
train_validation_df = load_fsd50k_split(
    FSD50K_GROUND_TRUTH_DIR / 'dev.csv',
    FSD50K_FEATURE_DIR, # This is for npy features
    split_overrides={'train': 'train', 'val': 'validation'},
)

test_df = load_fsd50k_split(
    FSD50K_GROUND_TRUTH_DIR / 'eval.csv',
    LOCAL_EVAL_NPY_FSD50K_PATH,
    split_overrides={None: 'test'},
)

fsd50k_df = pd.concat([train_validation_df, test_df], ignore_index=True)
assert not fsd50k_df.empty, 'No FSD50K examples were found. Check your feature or audio paths.'
assert fsd50k_df['target'].map(lambda target: target.sum() > 0).all(), 'Every example must have at least one known FSD50K label.'

split_summary = fsd50k_df.groupby('split_name').size().reindex(['train', 'validation', 'test'])
label_frequency = np.stack(fsd50k_df['target'].to_numpy()).sum(axis=0)

print({
    'num_classes': len(LABEL_ORDER),
    'split_counts': split_summary.fillna(0).astype(int).to_dict(),
    'available_clips': int(len(fsd50k_df)),
    'nonzero_labels': int(np.count_nonzero(label_frequency)),
})
if test_df.empty:
    print('⚠️ WARNING: No eval files detected. Training/validation will run, but test evaluation will be skipped.')

print('\nSplit summary:')
print(split_summary)


--- Debugging load_fsd50k_split for dev.csv ---
First 5 generated data_paths (before existence check):
  /npy_fsd50k/npy_fsd50k/64760.npy
  /npy_fsd50k/npy_fsd50k/16399.npy
  /npy_fsd50k/npy_fsd50k/16401.npy
  /npy_fsd50k/npy_fsd50k/16402.npy
  /npy_fsd50k/npy_fsd50k/16404.npy
--- End Debugging ---


--- Debugging load_fsd50k_split for eval.csv ---
First 5 generated data_paths (before existence check):
  /npy_fsd50k_eval/npy_fsd50k_eval/37199.npy
  /npy_fsd50k_eval/npy_fsd50k_eval/175151.npy
  /npy_fsd50k_eval/npy_fsd50k_eval/253463.npy
  /npy_fsd50k_eval/npy_fsd50k_eval/329838.npy
  /npy_fsd50k_eval/npy_fsd50k_eval/1277.npy
--- End Debugging ---

{'num_classes': 200, 'split_counts': {'train': 36796, 'validation': 4170, 'test': 10231}, 'available_clips': 51197, 'nonzero_labels': 200}

Split summary:
split_name
train         36796
validation     4170
test          10231
dtype: int64


## Step 5: Audio Processing Functions

In [ ]:
def load_feature_array(feature_path: tf.Tensor) -> tf.Tensor:
    path = feature_path.numpy().decode('utf-8')
    feature = np.load(path)
    feature = np.asarray(feature, dtype=np.float32)

    # Normalise the array shape so downstream resizing always sees a 2D mel image.
    if feature.ndim == 1:
        feature = np.expand_dims(feature, axis=0)
    if feature.ndim != 2:
        feature = np.squeeze(feature)
        if feature.ndim != 2:
            raise ValueError(f'Unexpected feature shape: {feature.shape} for {path}')

    expected_mels = PREPROCESSING_CONFIG['mel']['n_mels']
    if feature.shape[0] == expected_mels:
        return feature
    if feature.shape[1] == expected_mels:
        return feature.T

    raise ValueError(
        f'Unexpected mel shape {feature.shape} for {path}; expected one axis to be {expected_mels} bins.'
    )


def mel_feature_to_rgb_image(feature: tf.Tensor) -> tf.Tensor:
    feature = tf.convert_to_tensor(feature, dtype=tf.float32)
    feature = tf.expand_dims(feature, axis=-1)
    feature = tf.image.resize(feature, IMAGE_SIZE)
    feature = tf.repeat(feature, repeats=3, axis=-1)
    feature = tf.image.per_image_standardization(feature)
    return feature


def load_example(data_path: tf.Tensor, target: tf.Tensor):
    feature = tf.py_function(load_feature_array, [data_path], Tout=tf.float32)
    feature.set_shape([None, None])
    image = mel_feature_to_rgb_image(feature)
    image.set_shape([IMAGE_SIZE[0], IMAGE_SIZE[1], 3])
    target.set_shape([len(LABEL_ORDER)])
    return image, target

print('Numpy-feature processing functions defined')


Numpy-feature processing functions defined


## Step 6: Create Datasets

In [ ]:
def make_dataset(split_name: str, training: bool) -> tf.data.Dataset:
    frame = fsd50k_df[fsd50k_df['split_name'] == split_name].copy()
    assert not frame.empty, f'No examples available for split: {split_name}'
    targets = np.stack(frame['target'].to_numpy()).astype(np.float32)
    # Changed 'feature_path' to 'data_path' as per load_fsd50k_split modification
    dataset = tf.data.Dataset.from_tensor_slices((frame['data_path'].to_numpy(), targets))

    if training:
        dataset = dataset.shuffle(buffer_size=len(frame), seed=42, reshuffle_each_iteration=True)
        options = tf.data.Options()
        options.experimental_deterministic = False
        dataset = dataset.with_options(options)

    dataset = dataset.map(lambda data_path, target: load_example(data_path, target), num_parallel_calls=AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE, drop_remainder=training).prefetch(AUTOTUNE)
    return dataset

train_ds = make_dataset('train', training=True)
validation_ds = make_dataset('validation', training=False)
test_ds = make_dataset('test', training=False) if (fsd50k_df['split_name'] == 'test').any() else None

print('Datasets created')
print(f'Train dataset: {train_ds}')
print(f'Validation dataset: {validation_ds}')
print(f'Test dataset: {test_ds}')

Datasets created
Train dataset: <_PrefetchDataset element_spec=(TensorSpec(shape=(32, 256, 256, 3), dtype=tf.float32, name=None), TensorSpec(shape=(32, 200), dtype=tf.float32, name=None))>
Validation dataset: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 200), dtype=tf.float32, name=None))>
Test dataset: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 200), dtype=tf.float32, name=None))>


## Step 7: Build Model

In [ ]:
def build_mobilenetv4_classifier(num_classes: int) -> tuple[keras.Model, keras.Model]:
    inputs = keras.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3), name='log_mel_rgb')
    backbone = mobilenet.MobileNet(
        model_id=MODEL_ID,
        input_specs=keras.layers.InputSpec(shape=[None, IMAGE_SIZE[0], IMAGE_SIZE[1], 3]),
        stochastic_depth_drop_rate=0.1,
        output_stride=32,
        name='mobilenetv4_backbone',
    )
    backbone.trainable = True
    endpoints = backbone(inputs)
    final_endpoint_key = sorted([key for key in endpoints if key.isdigit()], key=int)[-1]
    x = endpoints[final_endpoint_key]
    if len(x.shape) == 4:
        x = keras.layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = keras.layers.Dropout(0.2, name='pre_embedding_dropout')(x)
    embedding = keras.layers.Dense(EMBEDDING_DIM, name='embedding')(x)
    normalized_embedding = keras.layers.Lambda(
        lambda tensor: tf.math.l2_normalize(tensor, axis=-1),
        name='embedding_norm',
    )(embedding)
    outputs = keras.layers.Dense(
        num_classes,
        activation='sigmoid',
        dtype='float32',
        name='fsd50k_multilabel_classifier',
    )(normalized_embedding)
    classifier = keras.Model(inputs=inputs, outputs=outputs, name='soundpulse_mobilenetv4_fsd50k_classifier')
    embedding_model = keras.Model(inputs=inputs, outputs=normalized_embedding, name='soundpulse_mobilenetv4_embedding_model')
    return classifier, embedding_model

classifier, embedding_model = build_mobilenetv4_classifier(len(LABEL_ORDER))
classifier.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY),
    loss='binary_crossentropy',
    metrics=[
        keras.metrics.BinaryAccuracy(name='binary_accuracy', threshold=PREDICTION_THRESHOLD),
        keras.metrics.AUC(name='micro_auprc', curve='PR', multi_label=False),
    ],
)
print('Model built and compiled')
print(f'Model ID: {MODEL_ID}')

Model built and compiled
Model ID: MobileNetV4ConvSmall


## Step 8: Train Model

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_OUTPUT_DIR / 'best.weights.h5'),
        monitor='val_micro_auprc',
        mode='max',
        save_best_only=True,
        save_weights_only=True,
    ),
    keras.callbacks.EarlyStopping(monitor='val_micro_auprc', mode='max', patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_micro_auprc', mode='max', factor=0.5, patience=2, min_lr=1e-6),
]

history = classifier.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

print('\n✓ Training completed!')

Epoch 1/20
1149/1149 [==============================] - 347s 147ms/step - loss: 0.2059 - binary_accuracy: 0.9846 - micro_auprc: 0.0190 - val_loss: 0.1537 - val_binary_accuracy: 0.9842 - val_micro_auprc: 0.0340 - lr: 3.0000e-04
Epoch 2/20
1149/1149 [==============================] - 210s 182ms/step - loss: 0.3697 - binary_accuracy: 0.9863 - micro_auprc: 0.0498 - val_loss: 0.4324 - val_binary_accuracy: 0.9842 - val_micro_auprc: 0.1014 - lr: 3.0000e-04
Epoch 3/20
1149/1149 [==============================] - 131s 114ms/step - loss: 0.3723 - binary_accuracy: 0.9863 - micro_auprc: 0.0975 - val_loss: 0.3291 - val_binary_accuracy: 0.9842 - val_micro_auprc: 0.1070 - lr: 3.0000e-04
Epoch 4/20
1149/1149 [==============================] - 132s 115ms/step - loss: 0.2554 - binary_accuracy: 0.9863 - micro_auprc: 0.1084 - val_loss: 0.2338 - val_binary_accuracy: 0.9842 - val_micro_auprc: 0.1083 - lr: 3.0000e-04
Epoch 5/20
1149/1149 [==============================] - 151s 131ms/step - loss: 0.2114 - bin

## Step 9: Evaluate Model

In [ ]:
if test_ds is None:
    metrics = {'status': 'skipped', 'reason': 'No eval .npy features found'}
    with (MODEL_OUTPUT_DIR / 'fsd50k_test_metrics.json').open('w', encoding='utf-8') as handle:
        json.dump(metrics, handle, indent=2)
    print('Test evaluation skipped: no test dataset available.')
else:
    test_probabilities = classifier.predict(test_ds)
    test_targets = np.concatenate([targets.numpy() for _, targets in test_ds], axis=0)
    test_predictions = (test_probabilities >= PREDICTION_THRESHOLD).astype(np.float32)

    metrics = {
        'micro_f1': float(f1_score(test_targets, test_predictions, average='micro', zero_division=0)),
        'macro_f1': float(f1_score(test_targets, test_predictions, average='macro', zero_division=0)),
        'micro_average_precision': float(average_precision_score(test_targets, test_probabilities, average='micro')),
        'macro_average_precision': float(average_precision_score(test_targets, test_probabilities, average='macro')),
        'label_ranking_average_precision': float(label_ranking_average_precision_score(test_targets, test_probabilities)),
        'hamming_loss': float(hamming_loss(test_targets, test_predictions)),
    }

    with (MODEL_OUTPUT_DIR / 'fsd50k_test_metrics.json').open('w', encoding='utf-8') as handle:
        json.dump(metrics, handle, indent=2)

    print('Test Metrics:')
    for metric, value in metrics.items():
        print(f'{metric}: {value:.4f}')

320/320 [==============================] - 97s 150ms/step
Test Metrics:
micro_f1: 0.0280
macro_f1: 0.0029
micro_average_precision: 0.1134
macro_average_precision: 0.0327
label_ranking_average_precision: 0.2169
hamming_loss: 0.0187


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a DataFrame for easier plotting
label_counts_df = pd.DataFrame({
    'label': LABEL_ORDER,
    'frequency': label_frequency
})

# Sort by frequency to easily see underrepresented classes
label_counts_df = label_counts_df.sort_values('frequency', ascending=True)

# Plot the label distribution
plt.figure(figsize=(15, 8))
sns.barplot(x='frequency', y='label', data=label_counts_df.head(20), palette='viridis')
plt.title('Top 20 Least Frequent Labels (Underrepresented Classes)')
plt.xlabel('Frequency')
plt.ylabel('Label')
plt.tight_layout()
plt.show()

/tmp/ipykernel_559/3393307634.py:15: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='frequency', y='label', data=label_counts_df.head(20), palette='viridis')


In [ ]:
plt.figure(figsize=(15, 8))
sns.barplot(x='frequency', y='label', data=label_counts_df.tail(20), palette='magma')
plt.title('Top 20 Most Frequent Labels')
plt.xlabel('Frequency')
plt.ylabel('Label')
plt.tight_layout()
plt.show()

/tmp/ipykernel_559/2369452329.py:2: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='frequency', y='label', data=label_counts_df.tail(20), palette='magma')


## Step 10: Save Model Artifacts

In [ ]:
# Collect embeddings
def collect_embeddings(dataset: tf.data.Dataset) -> tuple[np.ndarray, np.ndarray]:
    embeddings = []
    targets = []
    for batch_images, batch_targets in dataset:
        embeddings.append(embedding_model.predict(batch_images, verbose=0))
        targets.append(batch_targets.numpy())
    return np.concatenate(embeddings, axis=0), np.concatenate(targets, axis=0)

if test_ds is None:
    print('Embedding export skipped: no test dataset available.')
else:
    test_embeddings, test_embedding_targets = collect_embeddings(test_ds)
    np.save(MODEL_OUTPUT_DIR / 'test_embeddings.npy', test_embeddings)
    np.save(MODEL_OUTPUT_DIR / 'test_targets.npy', test_embedding_targets)

    print(f'Test embeddings shape: {test_embeddings.shape}')

Embedding export skipped: no test dataset available.


In [ ]:
# Save models
classifier.save(MODEL_OUTPUT_DIR / 'classifier.keras')
embedding_model.save(MODEL_OUTPUT_DIR / 'embedding_model.keras')
embedding_model.export(str(MODEL_OUTPUT_DIR / 'saved_model'))

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_saved_model(str(MODEL_OUTPUT_DIR / 'saved_model'))
# Add the following line to allow TensorFlow operations (FlexDelegate)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
tflite_model = converter.convert()
with (MODEL_OUTPUT_DIR / 'mobilenetv4_conv_small_embedding.tflite').open('wb') as handle:
    handle.write(tflite_model)

# Save metadata
metadata = {
    'model_name': MODEL_ID,
    'training_mode': 'from_scratch',
    'dataset': 'FSD50K',
    'task': 'multi_label_audio_classification',
    'num_classes': len(LABEL_ORDER),
    'label_order': LABEL_ORDER,
    'embedding_dim': EMBEDDING_DIM,
    'image_size': IMAGE_SIZE,
    'sample_rate_hz': SAMPLE_RATE,
    'window_duration_ms': WINDOW_DURATION_MS,
    'prediction_threshold': PREDICTION_THRESHOLD,
}
with (MODEL_OUTPUT_DIR / 'export_metadata.json').open('w', encoding='utf-8') as handle:
    json.dump(metadata, handle, indent=2)

print(f'✓ Models saved to {MODEL_OUTPUT_DIR}')

NameError: name 'classifier' is not defined

## Step 11: Copy Results to Google Drive

In [ ]:
import shutil

print(f'Copying results from {MODEL_OUTPUT_LOCAL_PATH} to {MODEL_OUTPUT_DRIVE_PATH}...')
# Copy all files
for item in MODEL_OUTPUT_LOCAL_PATH.iterdir():
    if item.is_file():
        shutil.copy2(item, MODEL_OUTPUT_DRIVE_PATH / item.name)
    elif item.is_dir():
        target_dir = MODEL_OUTPUT_DRIVE_PATH / item.name
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.copytree(item, target_dir)

print(f'✓ Results saved to Google Drive: {MODEL_OUTPUT_DRIVE_PATH}')
print('Files:')
for f in MODEL_OUTPUT_DRIVE_PATH.rglob('*'):
    if f.is_file():
        print(f'  {f.relative_to(MODEL_OUTPUT_DRIVE_PATH)}')

Copying results from /content/model_outputs to /content/drive/MyDrive/SoundPulse_outputs/mobilenetv4_conv_small_fsd50k_pretrain...
✓ Results saved to Google Drive: /content/drive/MyDrive/SoundPulse_outputs/mobilenetv4_conv_small_fsd50k_pretrain
Files:
  fsd50k_test_metrics.json
  best.weights.h5
  classifier.keras
  embedding_model.keras
  saved_model/saved_model.pb
  saved_model/fingerprint.pb
  saved_model/variables/variables.index
  saved_model/variables/variables.data-00000-of-00001
